In [1]:
import pandas as pd

In [9]:
import os
import json
import warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)

train = pd.read_csv("train.csv")
DROP = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
features = [c for c in train.columns if c not in DROP]

y = np.log1p(train["hours_spent"].values)
h_true = train["hours_spent"].values
strata = train["hours_bucket"].fillna("unk").astype(str)

# 1. CatBoost feature matrix
X_cb = train[features].copy()
cat_cols = []
for c in features:
    if X_cb[c].dtype == bool or str(X_cb[c].dtype) == "boolean":
        X_cb[c] = X_cb[c].astype("float32")
    elif not pd.api.types.is_numeric_dtype(X_cb[c]):
        cat_cols.append(c)
        fill_v = "other" if c == "project_category" else ""
        X_cb[c] = X_cb[c].fillna(fill_v).astype(str)
    else:
        X_cb[c] = pd.to_numeric(X_cb[c], errors="coerce").astype("float32")

# 2. XGBoost feature matrix (pure string categories + float32 numerics/booleans)
X_xgb = train[features].copy()
bool_cols = [c for c in features if X_xgb[c].dtype == bool or str(X_xgb[c].dtype) == "boolean"]
for c in bool_cols:
    X_xgb[c] = pd.to_numeric(X_xgb[c].astype("float32"), errors="coerce")
xgb_cats = {}
for c in cat_cols:
    s = X_xgb[c].dropna().astype(str).reindex(X_xgb.index)
    X_xgb[c] = s.astype("category")
    xgb_cats[c] = list(X_xgb[c].cat.categories)
for c in features:
    if c not in bool_cols and c not in cat_cols:
        X_xgb[c] = pd.to_numeric(X_xgb[c], errors="coerce").astype("float32")

SEEDS = (42, 2026, 7)
oof_cb7 = np.zeros(len(train))
oof_cb8 = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

idx = 0
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (tr_i, va_i) in enumerate(skf.split(X_cb, strata)):
        cb7 = CatBoostRegressor(
            iterations=2200, learning_rate=0.015, depth=7, l2_leaf_reg=4.0,
            subsample=0.8, bootstrap_type="Bernoulli", cat_features=cat_cols,
            random_seed=seed + fold, verbose=0, allow_writing_files=False
        ).fit(X_cb.iloc[tr_i], y[tr_i])
        cb7.save_model(f"models/cb7_{idx}.cbm")
        oof_cb7[va_i] += cb7.predict(X_cb.iloc[va_i]) / len(SEEDS)

        cb8 = CatBoostRegressor(
            iterations=2200, learning_rate=0.015, depth=8, l2_leaf_reg=6.0,
            subsample=0.8, bootstrap_type="Bernoulli", cat_features=cat_cols,
            random_seed=seed + fold + 100, verbose=0, allow_writing_files=False
        ).fit(X_cb.iloc[tr_i], y[tr_i])
        cb8.save_model(f"models/cb8_{idx}.cbm")
        oof_cb8[va_i] += cb8.predict(X_cb.iloc[va_i]) / len(SEEDS)

        xm = XGBRegressor(
            n_estimators=1600, learning_rate=0.012, max_depth=4, min_child_weight=30,
            subsample=0.8, colsample_bytree=0.5, reg_alpha=1.5, reg_lambda=8.0,
            enable_categorical=True, tree_method="hist",
            random_state=seed + fold, n_jobs=-1, verbosity=0
        ).fit(X_xgb.iloc[tr_i], y[tr_i])
        xm.save_model(f"models/xgb_{idx}.json")
        oof_xgb[va_i] += xm.predict(X_xgb.iloc[va_i]) / len(SEEDS)

        idx += 1
        print(f"Trained fold {idx}/15")

M = np.column_stack([oof_cb7, oof_cb8, oof_xgb])
res = minimize(
    lambda w: root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(M @ (w / np.sum(w))))),
    [0.45, 0.45, 0.10], bounds=[(0, 1)] * 3
)
weights = (res.x / np.sum(res.x)).tolist()
print("Optimal weights [cb7, cb8, xgb]:", [round(w, 4) for w in weights])
print("3-Seed Out-of-Fold CV RMSLE:", round(res.fun, 6))

with open("models/meta.json", "w") as f:
    json.dump({
        "features": features,
        "cat_cols": cat_cols,
        "bool_cols": bool_cols,
        "xgb_cats": xgb_cats,
        "n_models": idx,
        "weights": weights,
    }, f)
print("Saved all 45 models + models/meta.json successfully!")


Trained fold 1/15
Trained fold 2/15
Trained fold 3/15
Trained fold 4/15
Trained fold 5/15
Trained fold 6/15
Trained fold 7/15
Trained fold 8/15
Trained fold 9/15
Trained fold 10/15
Trained fold 11/15
Trained fold 12/15
Trained fold 13/15
Trained fold 14/15
Trained fold 15/15
Optimal weights [cb7, cb8, xgb]: [0.2083, 0.3367, 0.455]
3-Seed Out-of-Fold CV RMSLE: 0.654997
Saved all 45 models + models/meta.json successfully!


In [10]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import optuna
import joblib
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.makedirs("models_v2", exist_ok=True)

train = pd.read_csv("train.csv")
DROP = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
all_feats = [c for c in train.columns if c not in DROP]

y = np.log1p(train["hours_spent"].values)
h_true = train["hours_spent"].values
strata = train["hours_bucket"].fillna("unk").astype(str)


def prep_tables(df, feat_list):
    X_cb = df[feat_list].copy()
    cat_cols, bool_cols, num_cols = [], [], []
    for c in feat_list:
        if X_cb[c].dtype == bool or str(X_cb[c].dtype) == "boolean":
            bool_cols.append(c)
            X_cb[c] = pd.to_numeric(X_cb[c], errors="coerce").astype("float32")
        elif not pd.api.types.is_numeric_dtype(X_cb[c]):
            cat_cols.append(c)
            fill_v = "other" if c == "project_category" else ""
            X_cb[c] = X_cb[c].fillna(fill_v).astype(str)
        else:
            num_cols.append(c)
            X_cb[c] = pd.to_numeric(X_cb[c], errors="coerce").astype("float32")
    return X_cb, cat_cols, bool_cols, num_cols


print("1/4: Checking feature importance & dropping garbage columns...")
X_all, cat_cols_all, _, _ = prep_tables(train, all_feats)
skf5 = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_all, strata))

imp_scores = np.zeros(len(all_feats))
for fold, (tr_i, va_i) in enumerate(skf5):
    m = CatBoostRegressor(
        iterations=1200, learning_rate=0.025, depth=6, l2_leaf_reg=5.0,
        cat_features=cat_cols_all, random_seed=42 + fold, verbose=0, allow_writing_files=False
    ).fit(X_all.iloc[tr_i], y[tr_i])
    imp_scores += m.get_feature_importance() / 5.0

feat_imp = pd.Series(imp_scores, index=all_feats).sort_values(ascending=False)
keep_feats = feat_imp[feat_imp > 0.05].index.tolist()
print(f"Kept {len(keep_feats)}/{len(all_feats)} features (dropped {len(all_feats) - len(keep_feats)} noisy columns)")

X_cb, cat_cols, bool_cols, num_cols = prep_tables(train, keep_feats)

print("\n2/4: Running 30 Optuna trials to find best CatBoost settings...")
def objective(trial):
    params = {
        "iterations": 1800,
        "learning_rate": trial.suggest_float("learning_rate", 0.012, 0.03, log=True),
        "depth": trial.suggest_int("depth", 6, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 12.0),
        "random_strength": trial.suggest_float("random_strength", 0.5, 4.0),
        "subsample": trial.suggest_float("subsample", 0.65, 0.90),
        "one_hot_max_size": trial.suggest_categorical("one_hot_max_size", [2, 5, 10, 20]),
        "bootstrap_type": "Bernoulli",
        "cat_features": cat_cols,
        "verbose": 0,
        "allow_writing_files": False,
    }
    oof = np.zeros(len(X_cb))
    for fold, (tr_i, va_i) in enumerate(skf5):
        m = CatBoostRegressor(**params, random_seed=42 + fold).fit(X_cb.iloc[tr_i], y[tr_i])
        oof[va_i] = m.predict(X_cb.iloc[va_i])
    return root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(oof)))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
best_cb_params = dict(study.best_params)
best_cb_params.update({
    "iterations": 2400,
    "bootstrap_type": "Bernoulli",
    "cat_features": cat_cols,
    "verbose": 0,
    "allow_writing_files": False,
})
print("Best Optuna 5-Fold CV:", round(study.best_value, 6))
print("Best Params:", study.best_params)

print("\n3/4: Training 10-Fold Ensemble (CatBoost RMSE + CatBoost Huber + LightGBM + Neural Net)...")
skf10 = list(StratifiedKFold(n_splits=10, shuffle=True, random_state=42).split(X_cb, strata))

X_lgb = X_cb.copy()
for c in cat_cols:
    X_lgb[c] = X_lgb[c].astype("category")

oof_cb_rmse = np.zeros(len(X_cb))
oof_cb_huber = np.zeros(len(X_cb))
oof_lgb = np.zeros(len(X_cb))
oof_mlp = np.zeros(len(X_cb))

for fold, (tr_i, va_i) in enumerate(skf10):
    s = 100 + fold
    m1 = CatBoostRegressor(**best_cb_params, random_seed=s).fit(X_cb.iloc[tr_i], y[tr_i])
    m1.save_model(f"models_v2/cb_rmse_{fold}.cbm")
    oof_cb_rmse[va_i] = m1.predict(X_cb.iloc[va_i])

    huber_params = dict(best_cb_params)
    huber_params["loss_function"] = "Huber:delta=1.0"
    m2 = CatBoostRegressor(**huber_params, random_seed=s + 50).fit(X_cb.iloc[tr_i], y[tr_i])
    m2.save_model(f"models_v2/cb_huber_{fold}.cbm")
    oof_cb_huber[va_i] = m2.predict(X_cb.iloc[va_i])

    m3 = lgb.LGBMRegressor(
        n_estimators=1600, learning_rate=0.012, num_leaves=15, min_child_samples=40,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
        reg_alpha=1.5, reg_lambda=8.0, random_state=s, verbose=-1
    ).fit(X_lgb.iloc[tr_i], y[tr_i])
    joblib.dump(m3, f"models_v2/lgb_{fold}.pkl")
    oof_lgb[va_i] = m3.predict(X_lgb.iloc[va_i])

    pre = ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("qt", QuantileTransformer(n_quantiles=500, output_distribution="normal", random_state=s))
        ]), num_cols + bool_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), cat_cols),
    ])
    mlp = Pipeline([
        ("pre", pre),
        ("net", MLPRegressor(hidden_layer_sizes=(128, 64), alpha=5.0, max_iter=120, early_stopping=True, random_state=s))
    ]).fit(X_cb.iloc[tr_i], y[tr_i])
    joblib.dump(mlp, f"models_v2/mlp_{fold}.pkl")
    oof_mlp[va_i] = mlp.predict(X_cb.iloc[va_i])

    print(f"  Completed Fold {fold + 1}/10")

print("\n4/4: Training Out-of-Fold Ridge Meta-Stacker...")
M = np.column_stack([oof_cb_rmse, oof_cb_huber, oof_lgb, oof_mlp])
for name, p in [("CatBoost RMSE", oof_cb_rmse), ("CatBoost Huber", oof_cb_huber), ("LightGBM", oof_lgb), ("Tabular MLP", oof_mlp)]:
    print(f"  {name:<16} 10-Fold CV: {root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(p))):.6f}")

meta = RidgeCV(alphas=np.logspace(-1, 4, 50)).fit(M, y)
stack_pred = meta.predict(M)
print(f"  FINAL STACKED    10-Fold CV: {root_mean_squared_log_error(h_true, np.maximum(0.05, np.expm1(stack_pred))):.6f}")
print("  Stacker Coefs:", np.round(meta.coef_, 4), "Intercept:", round(float(meta.intercept_), 4))

joblib.dump(meta, "models_v2/meta_stacker.pkl")
with open("models_v2/config.json", "w") as f:
    json.dump({"keep_feats": keep_feats, "cat_cols": cat_cols, "bool_cols": bool_cols, "num_cols": num_cols}, f)
print("ALL DONE! Saved models_v2/ successfully.")


1/4: Checking feature importance & dropping garbage columns...
Kept 64/69 features (dropped 5 noisy columns)

2/4: Running 30 Optuna trials to find best CatBoost settings...
Best Optuna 5-Fold CV: 0.659724
Best Params: {'learning_rate': 0.02223114309253662, 'depth': 8, 'l2_leaf_reg': 5.416645161999993, 'random_strength': 1.1264631672995054, 'subsample': 0.8535532431581323, 'one_hot_max_size': 10}

3/4: Training 10-Fold Ensemble (CatBoost RMSE + CatBoost Huber + LightGBM + Neural Net)...
  Completed Fold 1/10
  Completed Fold 2/10
  Completed Fold 3/10
  Completed Fold 4/10
  Completed Fold 5/10
  Completed Fold 6/10
  Completed Fold 7/10
  Completed Fold 8/10
  Completed Fold 9/10
  Completed Fold 10/10

4/4: Training Out-of-Fold Ridge Meta-Stacker...
  CatBoost RMSE    10-Fold CV: 0.657278
  CatBoost Huber   10-Fold CV: 0.663098
  LightGBM         10-Fold CV: 0.661017
  Tabular MLP      10-Fold CV: 0.716504
  FINAL STACKED    10-Fold CV: 0.654574
  Stacker Coefs: [0.3727 0.2523 0.3345